# Notebook 11 — SPARQL Querying
Goal: Learn how to query RDF graphs using SPARQL with `rdflib`.

## Section 1 — Import Libraries
We will use `rdflib` to build a small RDF graph and run SPARQL queries against it.

In [ ]:
# If needed:
# pip install rdflib

from rdflib import Graph, Namespace, Literal

## Section 2 — Create a Namespace
This creates a prefix for building RDF identifiers. For example, `BIO.KRAS` expands to a full URI under the `http://example.org/bio/` namespace.

In [ ]:
BIO = Namespace("http://example.org/bio/")
BIO

## Section 3 — Build a Small RDF Graph
We will add genes, pathways, drugs, and patients as RDF triples.

In [ ]:
g = Graph()

g.add((BIO.KRAS, BIO.participates_in, BIO.MAPK_pathway))
g.add((BIO.BRAF, BIO.participates_in, BIO.MAPK_pathway))
g.add((BIO.EGFR, BIO.participates_in, BIO.EGFR_signaling))

g.add((BIO.Trametinib, BIO.targets, BIO.MEK1))
g.add((BIO.Cetuximab, BIO.targets, BIO.EGFR))
g.add((BIO.Erlotinib, BIO.targets, BIO.EGFR))

g.add((BIO.Patient1, BIO.has_mutation, BIO.KRAS))
g.add((BIO.Patient2, BIO.has_mutation, BIO.TP53))
g.add((BIO.Patient3, BIO.has_mutation, BIO.EGFR))

len(g)

## Section 4 — Inspect the Graph
Print the RDF graph in Turtle format to see what the triples look like.

In [ ]:
print(g.serialize(format="turtle"))

## Section 5 — What SPARQL Looks Like
A SPARQL query matches triple patterns. Variables start with `?`.

Example pattern:

```sparql
?gene bio:participates_in bio:MAPK_pathway .
```

## Section 6 — Query All Genes in the MAPK Pathway

In [ ]:
q1 = '''
PREFIX bio: <http://example.org/bio/>

SELECT ?gene
WHERE {
  ?gene bio:participates_in bio:MAPK_pathway .
}
'''

for row in g.query(q1):
    print(row)

## Section 7 — Query All Drugs Targeting EGFR

In [ ]:
q2 = '''
PREFIX bio: <http://example.org/bio/>

SELECT ?drug
WHERE {
  ?drug bio:targets bio:EGFR .
}
'''

for row in g.query(q2):
    print(row)

## Section 8 — Query All Patients with a KRAS Mutation

In [ ]:
q3 = '''
PREFIX bio: <http://example.org/bio/>

SELECT ?patient
WHERE {
  ?patient bio:has_mutation bio:KRAS .
}
'''

for row in g.query(q3):
    print(row)

## Section 9 — Use DISTINCT
`DISTINCT` removes duplicate results.

In [ ]:
q4 = '''
PREFIX bio: <http://example.org/bio/>

SELECT DISTINCT ?target
WHERE {
  ?drug bio:targets ?target .
}
'''

for row in g.query(q4):
    print(row)

## Section 10 — Use FILTER
`FILTER` lets you restrict results.

In [ ]:
q5 = '''
PREFIX bio: <http://example.org/bio/>

SELECT ?drug
WHERE {
  ?drug bio:targets ?target .
  FILTER(?target = bio:EGFR)
}
'''

for row in g.query(q5):
    print(row)

## Section 11 — Use COUNT
Aggregation is also possible in SPARQL.

In [ ]:
q6 = '''
PREFIX bio: <http://example.org/bio/>

SELECT ?target (COUNT(?drug) AS ?n_drugs)
WHERE {
  ?drug bio:targets ?target .
}
GROUP BY ?target
'''

for row in g.query(q6):
    print(row)

## Section 12 — Multi-Hop Query
This is where graph querying becomes powerful. We can connect patients to pathways by chaining relationships.

In [ ]:
q7 = '''
PREFIX bio: <http://example.org/bio/>

SELECT ?patient ?pathway
WHERE {
  ?patient bio:has_mutation ?gene .
  ?gene bio:participates_in ?pathway .
}
'''

for row in g.query(q7):
    print(row)

## Section 13 — Exercises
1. Add a triple linking `TP53` to `Cell_cycle`.
2. Write a query to find all genes with pathway annotations.
3. Write a query to find all patients connected to a pathway.
4. Write a query to count how many patients have mutations in each gene.
5. Add another drug-target relationship and re-run the aggregation query.

## Skills Gained
- understanding SPARQL query structure
- using `PREFIX`, `SELECT`, and `WHERE`
- querying RDF triple patterns
- using `DISTINCT`, `FILTER`, and `COUNT`
- writing multi-hop graph queries across biological relationships